In [ ]:
import glob, json, os, shutil, subprocess, time, zipfile
VERSION, BASE = '0.6.2', '0.6.1'   # new code vs the previous release, same inputs
AB = True   # True: timing A/B of VERSION vs BASE; False: OCR stage profile of VERSION only
W = '/kaggle/working'
for v in (VERSION, BASE):
    z = glob.glob(f'/kaggle/input/**/wmr-{v}.zip', recursive=True)
    d = glob.glob(f'/kaggle/input/**/wmr-{v}/wmr/track.py', recursive=True)
    if z:
        zipfile.ZipFile(z[0]).extractall(W)
    elif d:
        shutil.copytree(os.path.dirname(os.path.dirname(d[0])), f'{W}/wmr-{v}', dirs_exist_ok=True)
    else:
        raise SystemExit(f'wmr {v} not found in /kaggle/input')
print(sorted(os.listdir(W)))

In [ ]:
!nvidia-smi -L
!bash /kaggle/working/wmr-{VERSION}/scripts/setup_kaggle.sh 2>&1 | tail -4
!python /kaggle/working/wmr-{VERSION}/tests/test_fill_torch.py
!python /kaggle/working/wmr-{VERSION}/tests/test_rec_buckets.py
!python /kaggle/working/wmr-{VERSION}/tests/test_match_torch.py
!python /kaggle/working/wmr-{VERSION}/tests/test_lama.py

In [ ]:
# helpers: every result line also goes to results.txt (the Kaggle log truncates the middle)
os.chdir(W)
R = open(f'{W}/results.txt', 'w')
def say(s):
    print(s, flush=True); R.write(s + '\n'); R.flush()
KEEP = ('[analyze]', '[remove]', '[verify]', '[run]', 'note:', 'warning', 'track ', 'Error', 'Traceback')

def run(tag, ver, args, env=None):
    """python -m wmr <args> with that version's code (cwd = its folder); GPU utilisation sampled every 0.5 s."""
    smi = subprocess.Popen(['nvidia-smi', '--query-gpu=index,utilization.gpu', '--format=csv,noheader,nounits', '-lms', '500'],
                           stdout=subprocess.PIPE, text=True)
    t = time.time()
    r = subprocess.run(['python', '-m', 'wmr'] + args, cwd=f'{W}/wmr-{ver}', capture_output=True, text=True,
                       env={**os.environ, **(env or {})})
    el = time.time() - t
    smi.terminate(); out = smi.communicate()[0]
    util = {}
    for l in out.splitlines():
        try:
            g, u = (int(x) for x in l.split(','))
            util.setdefault(g, []).append(u)
        except ValueError:
            pass
    say(f'=== {tag} [{ver}{" " + str(env) if env else ""}]: {el:.1f} s wall, exit {r.returncode}; mean GPU util '
        + ', '.join(f'gpu{g} {sum(v) / len(v):.0f}%' for g, v in sorted(util.items())))
    for l in (r.stdout + r.stderr).splitlines():
        if any(k in l for k in KEEP):
            say('  ' + l.strip()[:420])
    return el

def ev(tag, clip, out):
    e = subprocess.run(['python', '-m', 'wmr', 'eval', clip, out, '--tracks', out[:-4] + '.tracks.json'],
                       cwd=f'{W}/wmr-{VERSION}', capture_output=True, text=True)
    say(f'  eval {tag}: ' + ' '.join(l.strip() for l in e.stdout.splitlines()
                                    if any(k in l for k in ('psnr_in_mask_after', 'psnr_ring', 'edge_err', 'recall_mean',
                                                           'flicker_in_mask_output'))))

def synth(name, *a):
    subprocess.run(['python', '-m', 'wmr', 'synth', '-o', f'{W}/{name}'] + list(a), cwd=f'{W}/wmr-{VERSION}',
                   check=True, capture_output=True)

synth('synth1', '--size', '1280x720', '--frames', '300', '--overlay', '@my_channel:bounce:1.0:40',
      '--overlay', 'SAMPLE MARK:static:0.4:40', '--overlay', 'hello2026:teleport:1.0:40', '--dwell', '100')
synth('long', '--size', '1920x1080', '--frames', '900', '--overlay', '@studio_mark:bounce:1.0:44',
      '--overlay', 'LIVE 2026:static:1.0:40', '--overlay', 'www.example-site.com:linear:0.5:30')
synth('hard', '--size', '1920x1080', '--frames', '300', '--overlay', '@tiny_user88:linear:0.3:22',
      '--overlay', 'www.example-site.com:static:0.25:26', '--overlay', 'ID 4417-QX:teleport:0.6:20', '--dwell', '40')
# four identical marks (one teleporting): tracks must not swap instances (0.5.5 regression on 072q-1)
synth('same', '--size', '1280x720', '--frames', '450', '--overlay', 'Huohua Studio:teleport:0.65:36',
      '--overlay', 'Huohua Studio:bounce:0.65:36', '--overlay', 'Huohua Studio:bounce:0.65:36',
      '--overlay', 'Huohua Studio:bounce:0.5:36', '--dwell', '60', '--speed', '6')
# shadowed (two-tone) marks over local motion and brightness drift: seams, smears, ghost outlines
synth('local', '--size', '1280x720', '--frames', '300', '--overlay', 'Huohua Studio:bounce:0.65:40:shadow',
      '--overlay', 'Studio Mark:static:0.65:40:shadow', '--overlay', 'hello2026:teleport:0.9:40:shadow',
      '--dwell', '75', '--blobs', '8', '--blob-radius', '60', '--exposure', '10')
say('synth clips ready')

In [ ]:
# OCR stage profile: where an OCR call spends its time on this machine's CPU and GPUs
if not AB:
    say(subprocess.run("nproc; lscpu | grep -E 'Model name|^CPU\\(s\\)|Thread|Core|MHz'; free -g | head -2",
                       shell=True, capture_output=True, text=True).stdout.strip())
    for clip in ('long',):
        p = subprocess.run(['python', 'scripts/ocr_profile.py', f'{W}/{clip}/marked.mp4', '24'],
                           cwd=f'{W}/wmr-{VERSION}', capture_output=True, text=True)
        say(f'### OCR profile: {clip}, exit {p.returncode}')
        for l in (p.stdout + p.stderr[-3000:]).splitlines():
            say(l)


In [ ]:
# synthetic A/B: same clips, new code vs previous release
if AB:
    for clip in ('local', 'same', 'synth1', 'hard', 'long'):
        for ver, extra, tag in ((VERSION, [], ''), (BASE, [], '')):
            out = f'{W}/{clip}/out_{ver}{tag}.mp4'
            run(clip + tag, ver, ['run', f'{W}/{clip}/marked.mp4', '-o', out, '--texts', f'{W}/{clip}/texts.txt'] + extra)
            ev(f'{clip} {ver}{tag}', f'{W}/{clip}', out)


In [ ]:
# keep only text results: no videos leave Kaggle
R.close()
for x in os.listdir(W):
    if x != 'results.txt':
        shutil.rmtree(os.path.join(W, x), ignore_errors=True) if os.path.isdir(os.path.join(W, x)) else os.remove(os.path.join(W, x))
!ls -la /kaggle/working